# İkinci el BMW ve Audi ilanları: açık analiz defteri

Bu defter projenin bütün analizlerini baştan sona, kodu açıkta anlatır. Her hesap pandas ve statsmodels adımlarıyla
hücrenin içinde durur. Her hücreden önce ne yapıldığı ve neden yapıldığı, sonra çıktının ne gösterdiği yazılıdır.
`notebooks/` altındaki öteki defterler (ör. [1_data.ipynb](../1_data.ipynb)) hesabı analiz betiğine bırakıp yalnız
sonucu gösterir; bu defter tersini yapar.

## Bu defter nasıl okunur

**Projenin sorusu.** İkinci el bir aracın ilan fiyatını ne belirler ve bir tahmin modeli bu fiyatı ne kadar isabetle
öngörebilir? Veri, bir ilan sitesinden dört ayrı günde toplanan BMW ve Audi otomobil ilanları. Hedef ilandaki fiyattır,
satış fiyatı değil: satış fiyatı pazarlıkla bundan ayrılır.

**Bölümler.**

| bölüm | konu |
|---|---|
| 0 | Kurulum ve veri: kütüphaneler, verinin yüklenmesi, tahmin modelinin girdileri |
| 1 | Veri temizliği: tarama satırından ilana, motor aralığından tek sayıya, "belirtilmemiş" hasar cevabı |
| 2 | Eksiklik: hangi kolonlar boş ve boşluk rastgele mi |
| 3 | Bağımlılık ve marka: kolonlar arası ilişki, segment, markanın katkısı |
| 4 | Hedef: fiyatın dağılımı ve neden logaritmasıyla çalışıldığı |
| 5 | Piyasa yapısı: ilanlar belirgin kümelere ayrılıyor mu |
| 6 | Hedonik regresyon: araç modeli sabitken yaşın, kilometrenin, hasarın ve motorun fiyata etkisi |
| 7 | Tahmin modeli: LightGBM ve rakipleri, çapraz doğrulamayla |
| 8 | Hata ve aralık: hatanın dağılımı, büyük hatalar, fiyat aralığı ve ileri test |
| 9 | Serbest metin: satıcının açıklaması tahmine bir şey katıyor mu |

**Ne burada hesaplanır, ne kayıttan okunur.** Hafif hesaplar (sayım, gruplama, regresyon) bu defterde, gözünüzün
önünde yapılır. Tahmin modeli eğiten koşumlar burada tekrarlanmaz: tahmin modelinin çapraz doğrulamayla eğitimi,
LOFO (öznitelikleri tek tek çıkarıp tahmin modelini yeniden eğitme testi) ve benzerleri dakikalar sürer. Bunların
sonucu, analiz betiklerinin `metrics/` altına yazdığı kayıtlardan okunur. Sonraki bölümler böyle her yeri
**Kayıttan okunur** diye işaretler. Bölüm 0 ve 1'de böyle bir yer yok: tahmin modeli eğitilmiyor, her sayı burada
hesaplanıyor.

**Kayıtla karşılaştırma.** Her analiz bir denetim hücresiyle biter. Hücre, burada hesaplanan ana sayıları o betiğin
kaydıyla (`metrics/<betik>.json`, raporların sayıları okuduğu dosya) `assert` ile karşılaştırır. Bir sayı tutmazsa
hücre hata verir ve defter durur. Böylece bu defter ikinci bir doğru üretmez: raporla aynı sayıları adım adım
gösterir.

**Kurallar.** Defter hiçbir dosya yazmaz; veritabanını ve ham dosyaları yalnız okur. Kod İngilizce adlarla yazılı
(kolon adları veritabanındaki gibi); Türkçe yalnız okura gösterilen metinde ve tablo başlıklarında. İlan numaraları
(`ad_id`) ve satıcı açıklamaları hiçbir çıktıda gösterilmez. Defter taze bir çekirdekte yukarıdan aşağı çalışır
(Restart & Run All); her hücre öncekilerin kurduğu tablolara dayanır.

> **Terimler**
>
> - **araç modeli**: ilanın `model` kolonu, yani aracın model adı (ör. "320i Sport Line").
> - **tahmin modeli**: ilan fiyatını tahmin eden LightGBM (Bölüm 7).
>
> İki anlam karışabileceği için bu defterde "model" kelimesi tek başına kullanılmaz; her yerde ya "araç modeli" ya
> "tahmin modeli" yazılır.
>
> - **tarama**: sitenin belirli bir günde baştan sona taranması. Dört tarama var.
> - **tarama satırı**: bir ilanın bir taramadaki kaydı. Aynı ilan birden çok taramada görülebilir.
> - **ilan**: tekil ilan, `ad_id` başına bir. Her ilan en son görüldüğü taramadaki satırıyla temsil edilir.
> - **öznitelik**: tahmin modelinin girdisi olan kolon.
> - **kayıt**: bir analiz betiğinin `metrics/` altına yazdığı JSON dosyası.

## 0. Kurulum ve veri

### 0.1 Kurulum

Aşağıdaki hücre kütüphaneleri yükler ve deponun kökünü bulur. Kök, içinde `analysis/lib` klasörü olan ilk üst
klasördür; defter hangi klasörden açılırsa açılsın aynı yere varılır. Sonra `analysis/` klasörü Python'un arama
yoluna eklenir: analiz betiklerinin ortak kodu (`analysis/lib`) böylece `lib` adıyla içe aktarılabilir.

In [1]:
import json
import re
import sys
from pathlib import Path

import duckdb
import pandas as pd

# Depo kökü: içinde analysis/lib klasörü olan ilk üst klasör
ROOT = Path.cwd()
while not (ROOT / "analysis" / "lib").is_dir():
    assert ROOT != ROOT.parent, "Depo kökü bulunamadı; defteri depo içinden açın."
    ROOT = ROOT.parent

# analysis/ arama yoluna eklenince analiz betiklerinin ortak kodu `lib` adıyla içe aktarılır
sys.path.insert(0, str(ROOT / "analysis"))

pd.set_option("display.max_rows", 30)
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 160)
pd.set_option("display.max_colwidth", 80)
print("Depo kökü bulundu:", ROOT.name)

Depo kökü bulundu: cardatasys


İki küçük yardımcı, defter boyunca birçok kez kullanılıyor: sayıyı Türkçe binlik ayırıcıyla yazan `tr_int` ve bir
analiz betiğinin kaydını okuyan `read_record`.

In [2]:
def tr_int(number):
    """Tam sayıyı Türkçe binlik ayırıcıyla yazar: 45159 -> '45.159'."""
    return f"{int(number):,}".replace(",", ".")


def read_record(script_name):
    """Bir analiz betiğinin kaydını (metrics/<betik>.json) okur."""
    return json.loads((ROOT / "metrics" / f"{script_name}.json").read_text(encoding="utf-8"))

### 0.2 Veriyi yükleme

Veri `data/cars.duckdb` dosyasındaki `car_listings` tablosunda: her satır bir ilanın bir taramadaki kaydı. Tablo,
toplanan ham dosyalardan `db/` klasöründeki kodla kurulur. Değerler sitede göründüğü gibi bırakılır (yarı ham), boş
olan boş kalır.

Analiz betikleri ve tahmin modeli bu tabloyu doğrudan okumaz; hepsi aynı kapıdan geçer: `lib.common.load_clean`
([common.py](../../analysis/lib/common.py)). Bu defter de onu kullanır, çünkü tahmin modelinin gördüğü satırlar ve
kolonlar tam olarak onun verdikleridir. `load_clean` şunları yapar:

1. **Satırları seçer:** fiyatı 0'dan büyük ve plakası TR olan satırlar (`gb_plate_origin = '(TR) Türkiye'`).
2. **İlanlara indirir:** `load_clean(all_snapshots=True)` bu satırların hepsini verir. Varsayılan `load_clean()` her
   `ad_id` için yalnız en son taramadaki satırı tutar, yani ilan başına bir satır verir. İki durumda da satırlar
   `ad_id` sırasındadır; çapraz doğrulamanın katları bu sıraya dayanır.
3. **Kolon türetir:**
   - `vehicle_age`: yaş, yani taramanın yılı eksi model yılı (`gb_year`); en az 0.
   - `power_hp_val` ve `engine_cc_val`: motor gücü ve hacmi tek sayı olarak. Site bunları bazı ilanlarda aralık
     olarak verir. Güç için aralığın iki sınırının ortalaması, hacim için üst sınırı alınır; kesin değerde iki
     sınır aynıdır. Veritabanında da bu adlarla kolon var (aralığın orta noktası); `load_clean` onların üstüne bu
     kuralla yazar. Kuralın gerekçesi Bölüm 1.2'de.
   - `is_heavy_damaged`: ağır hasar kaydı. Sayfa bilgi vermemişse 0, yani ağır hasarsız sayılır.
   - `segment`: aracın sınıfı (B, C, D, E, F; S spor araç). Seriden gelir: 3 Serisi ve A4 D, 5 Serisi ve A6 E gibi.
     Tek bir sınıfa düşmeyen performans aileleri (M, S, RS) araç modeli adındaki rakamdan çözülür: M3, 3 Serisi
     gibi D olur. i8 ve Z4 M spor araç sayılır. Çözülemeyen bir seri kalırsa yükleme durur
     ([segment_rule.py](../../analysis/lib/segment_rule.py)).
   - Hasar: site 13 kaporta parçasının her biri için bir cevap verir; veritabanında her parçanın üç bayrağı var
     (değişen, boyalı, lokal boyalı). Tek parça olan tavan, kaput ve bagaj için birer **durum** kolonu türetilir
     (`roof_state`, `hood_state`, `trunk_state`); öncelik sırası değişen, boyalı, lokal, orijinal. Kapı (4 parça),
     çamurluk (4) ve tampon (2) grupları için **sayı** kolonları türetilir: gruptaki kaç parça değişen, kaç parça
     boyalı, kaç parça lokal boyalı (ör. `door_painted`, boyalı kapı sayısı, 0-4).
   - `snap`: taramanın tarihi, metin olarak.
4. **Boşlukları işaretler:** metin ve kategorik özniteliklerde boş değer `"missing"` adlı ayrı bir kategori olur.
   Sayısal özniteliklerde boş değer boş kalır (NaN); tahmin modeli boşluğu kendisi işler.

Aşağıdaki hücre veriyi iki biçimde yükler.

In [3]:
from lib.common import CAT, DB_PATH, FEATURES, NUM, TEXT, load_clean

all_rows = load_clean(all_snapshots=True)   # her taramanın her TR plakalı satırı
listings = load_clean()                     # ilan başına bir satır: en son görüldüğü tarama
print("Tarama satırları (satır, kolon):", all_rows.shape)
print("İlanlar (satır, kolon):         ", listings.shape)

Tarama satırları (satır, kolon): (45159, 133)
İlanlar (satır, kolon):          (29988, 133)


`all_rows` bütün tarama satırlarını, `listings` ilanları tutuyor; kolonlar ikisinde de aynı. Bu defterin geri
kalanı çoğunlukla `listings` üzerinde çalışır.

Hangi taramalar var ve her birinde kaç satır? İkinci kolon, en son o taramada görülen ilan sayısı: `listings` her
ilanı son görüldüğü taramadan aldığı için ilanlar taramalara böyle dağılıyor.

In [4]:
snapshot_sizes = pd.DataFrame({
    "rows": all_rows["snap"].value_counts(),
    "listings_last_seen": listings["snap"].value_counts(),
}).sort_index()

snapshot_sizes.rename_axis("tarama").rename(columns={"rows": "tarama satırı",
                                                     "listings_last_seen": "en son bu taramada görülen ilan"})

,tarama satırı,en son bu taramada görülen ilan
tarama,,
2026-01-18,10901,2409
2026-01-27,11254,7002
2026-03-21,11478,9051
2026-06-27,11526,11526


**Ne görüyoruz.** Her taramada 11 bin civarı satır var. Son taramanın (2026-06-27) bütün satırları kendi ilanı,
çünkü ondan sonra tarama yok. İlk taramanın 10.901 satırından yalnız 2.409'u ilanın son görüldüğü yer; kalan
ilanlar sonraki taramalarda yeniden görülmüş. Satırdan ilana bu iniş Bölüm 1.1'de ayrıntılı.

`load_clean` ham satıra neyi ekliyor? Aynı satırlar türetme yapılmadan da istenebilir (`derived=False`). İki tabloyu
karşılaştırınca eklenen kolonlar görünür.

In [5]:
raw_listings = load_clean(derived=False)   # aynı satırlar, türetme yapılmadan
assert (raw_listings["ad_id"].values == listings["ad_id"].values).all()   # satır sırası aynı

added_columns = []
for column in listings.columns:
    if column not in raw_listings.columns:
        added_columns.append(column)
print(len(added_columns), "yeni kolon:", ", ".join(added_columns))

15 yeni kolon: vehicle_age, segment, roof_state, hood_state, trunk_state, door_changed, door_painted, door_local, fender_changed, fender_painted, fender_local, bumper_changed, bumper_painted, bumper_local, snap


Eklenen 15 kolonun yanında `load_clean` üç kolonun üstüne yazar: `engine_cc_val`, `power_hp_val` ve
`is_heavy_damaged`. Hacimde kural veritabanındaki değeri gerçekten değiştiriyor mu, güçte değiştirmiyor mu?

In [6]:
for column in ["engine_cc_val", "power_hp_val"]:
    new = listings[column]
    old = raw_listings[column]
    differs = (new != old) & new.notna() & old.notna()
    print(f"{column}: {tr_int(differs.sum())} ilanda veritabanındaki değerden farklı")

engine_cc_val: 7.827 ilanda veritabanındaki değerden farklı
power_hp_val: 0 ilanda veritabanındaki değerden farklı


**Ne görüyoruz.** Güçte `load_clean` veritabanıyla aynı değeri veriyor (iki sınırın ortalaması zaten aralığın orta
noktası). Hacimde 7.827 ilanın değeri değişiyor: hacmi iki sınırlı bir aralık olarak verilmiş ilanlar. Bunlarda orta
nokta yerine artık üst sınır kullanılıyor.

Türetilen kolonlardan örnekler: seri, araç modeli ve segment; model yılı, tarama ve yaş. Satırlar rastgele seçildi
(`random_state` sabit, her koşumda aynı satırlar).

In [7]:
age_and_segment = listings[["series", "model", "segment", "gb_year", "snap", "vehicle_age"]].sample(5, random_state=1)

age_and_segment.rename(columns={"series": "seri", "model": "araç modeli", "segment": "segment",
                                "gb_year": "model yılı", "snap": "tarama", "vehicle_age": "yaş"})

,seri,araç modeli,segment,model yılı,tarama,yaş
25457,4 Serisi,420d M Sport,D,2015,2026-06-27,11
6631,A5,A5 Cabrio 45 TFSI Quattro S Line,D,2023,2026-01-18,3
14528,A7,3.0 TDI,E,2016,2026-03-21,10
12785,A4,A4 Sedan 2.0 TDI,D,2013,2026-01-27,13
5674,5 Serisi,525d xDrive M Sport,E,2014,2026-03-21,12


Hasar sayısı nasıl kuruluyor? Aşağıda en az iki kapısı boyalı üç ilan var: dört kapının "boyalı" bayrağı
(veritabanından) ve onlardan sayılan `door_painted`. Boş bayrak (`<NA>`) sayılırken 0 kabul edilir.

In [8]:
door_flags = ["door_fl_boyali", "door_fr_boyali", "door_rl_boyali", "door_rr_boyali"]
door_example = listings.loc[listings["door_painted"] >= 2, door_flags + ["door_painted"]].head(3)

door_example.rename(columns={"door_fl_boyali": "sol ön kapı boyalı", "door_fr_boyali": "sağ ön kapı boyalı",
                             "door_rl_boyali": "sol arka kapı boyalı", "door_rr_boyali": "sağ arka kapı boyalı",
                             "door_painted": "boyalı kapı sayısı"})

,sol ön kapı boyalı,sağ ön kapı boyalı,sol arka kapı boyalı,sağ arka kapı boyalı,boyalı kapı sayısı
0,0,1,0,1,2
1,<NA>,1,<NA>,1,2
4,0,1,0,1,2


Tek parçaların durumu: tavan, kaput ve bagaj her ilanda hangi durumda?

In [9]:
state_counts = pd.DataFrame({
    "roof_state": listings["roof_state"].value_counts(),
    "hood_state": listings["hood_state"].value_counts(),
    "trunk_state": listings["trunk_state"].value_counts(),
}).reindex(["original", "local", "painted", "changed"])

state_counts.rename(columns={"roof_state": "tavan", "hood_state": "kaput", "trunk_state": "bagaj"},
                    index={"original": "orijinal", "local": "lokal boyalı", "painted": "boyalı", "changed": "değişen"})

,tavan,kaput,bagaj
orijinal,29647,22722,26756
lokal boyalı,85,1611,1067
boyalı,246,3107,1529
değişen,10,2548,636


**Ne görüyoruz.** Tavan neredeyse hiç işlem görmemiş (29.988 ilandan 29.647'sinde orijinal). Kaput en sık işlem
gören tek parça: her dört ilandan birinde orijinal değil ve 2.548 ilanda değişmiş.

### 0.3 Tahmin modelinin girdileri

`lib.common.FEATURES` tahmin modelinin kullandığı kolonların listesi. Üç parçadan oluşur: `TEXT` (metin olarak
işlenen adlar, Bölüm 7), `CAT` (kategorik kolonlar) ve `NUM` (sayısal kolonlar). Kolon adlarındaki `kb_` ve `gb_`
önekleri ilan sayfasının iki bölümünü gösterir: kısa bilgi kutusu ve genel bakış. Aynı bilgi iki yerde de varsa
biri seçildi (Bölüm 2).

Aşağıdaki sözlük her özniteliği bir satırla açıklar; sonraki hücre onu `FEATURES` ile karşılaştırıp tabloya çevirir.

In [10]:
FEATURE_NOTES = {
    "model": "araç modeli adı, ör. 320i Sport Line",
    "series": "seri adı, ör. 3 Serisi, A4",
    "brand": "marka: bmw ya da audi",
    "kb_body_type": "kasa tipi: Sedan, Hatchback/5, Coupe ...",
    "kb_drivetrain": "çekiş: arkadan itiş, önden çekiş, 4WD ...",
    "segment": "araç sınıfı B-F ya da S (spor); seriden türetildi",
    "kb_transmission": "vites: otomatik, yarı otomatik, düz",
    "kb_fuel": "yakıt: benzin, dizel, LPG & benzin, hibrit",
    "roof_state": "tavanın durumu: orijinal, lokal boyalı, boyalı ya da değişen",
    "hood_state": "motor kaputunun durumu (aynı dört değer)",
    "trunk_state": "bagaj kapağının durumu (aynı dört değer)",
    "vehicle_age": "yaş (yıl): tarama yılı eksi model yılı",
    "gb_mileage": "kilometre",
    "power_hp_val": "motor gücü (hp); aralıkta iki sınırın ortalaması",
    "engine_cc_val": "motor hacmi (cc); aralıkta üst sınır",
    "door_changed": "değişen kapı sayısı (0-4)",
    "door_painted": "boyalı kapı sayısı (0-4)",
    "door_local": "lokal boyalı kapı sayısı (0-4)",
    "fender_changed": "değişen çamurluk sayısı (0-4)",
    "fender_painted": "boyalı çamurluk sayısı (0-4)",
    "fender_local": "lokal boyalı çamurluk sayısı (0-4)",
    "bumper_changed": "değişen tampon sayısı (0-2)",
    "bumper_painted": "boyalı tampon sayısı (0-2)",
    "bumper_local": "lokal boyalı tampon sayısı (0-2)",
    "is_heavy_damaged": "ağır hasar kaydı: 1 var, 0 yok ya da bilinmiyor",
}

In [11]:
assert list(FEATURE_NOTES) == FEATURES, "açıklama listesi FEATURES ile aynı sırada olmalı"

feature_table = pd.DataFrame({
    "column": FEATURES,
    "group": ["TEXT"] * len(TEXT) + ["CAT"] * len(CAT) + ["NUM"] * len(NUM),
    "note": list(FEATURE_NOTES.values()),
})
feature_table["n_values"] = listings[FEATURES].nunique().values
print(len(FEATURES), "öznitelik:", len(TEXT), "metin,", len(CAT), "kategorik,", len(NUM), "sayısal")

feature_table.rename(columns={"column": "kolon", "group": "liste", "note": "ne", "n_values": "farklı değer"})

25 öznitelik: 2 metin, 9 kategorik, 14 sayısal


,kolon,liste,ne,farklı değer
0,model,TEXT,"araç modeli adı, ör. 320i Sport Line",745
1,series,TEXT,"seri adı, ör. 3 Serisi, A4",22
2,brand,CAT,marka: bmw ya da audi,2
3,kb_body_type,CAT,"kasa tipi: Sedan, Hatchback/5, Coupe ...",10
4,kb_drivetrain,CAT,"çekiş: arkadan itiş, önden çekiş, 4WD ...",5
5,segment,CAT,araç sınıfı B-F ya da S (spor); seriden türetildi,6
6,kb_transmission,CAT,"vites: otomatik, yarı otomatik, düz",3
7,kb_fuel,CAT,"yakıt: benzin, dizel, LPG & benzin, hibrit",4
8,roof_state,CAT,"tavanın durumu: orijinal, lokal boyalı, boyalı ya da değişen",4
9,hood_state,CAT,motor kaputunun durumu (aynı dört değer),4


**Ne görüyoruz.** Tahmin modeli 25 kolon görüyor. Kimlik iki metinle geliyor: 22 seri ve 745 farklı araç modeli
adı; araç modeli adı seriden çok daha ince bir ayrım taşıyor. Hasar, 13 parçanın 39 bayrağı yerine 12 kolonla
(3 durum, 9 sayı) ve ağır hasar kaydıyla giriyor. Kategorik kolonların değer sayısına `missing` kategorisi de
dahil.

## 1. Veri temizliği

Tahmin modeline geçmeden önce verinin kendisiyle ilgili üç soru:

1. **Tarama satırından ilana:** dört taramanın satırları tekil ilanlara nasıl iniyor, veri hangi sınırlarla
   toplandı ve aynı aracın iki ayrı ilan olarak görünmesi ne kadar sık?
2. **Motor aralığı:** site gücü ve hacmi aralık olarak verdiğinde tek sayı olarak ne kullanılmalı?
3. **"Belirtilmemiş" hasar cevabı:** orijinal sayılan bu cevap kaç parçada ve kaç ilanda var?

Her soru bir analiz betiğinin konusu. Burada her birinin hesabı betiği çağırmadan, adım adım yeniden yazılıyor.
Raporda karşılığı: [teknik rapor §1](../../reports/technical.tr.md).

### 1.1 Tarama satırından ilana ve sızıntı riski

Kaynak: [betik](../../analysis/01_dedup_leakage.py) · [kart](../../analysis/cards/01_dedup_leakage.json) ·
[kayıt](../../metrics/01_dedup_leakage.json)

**Soru.** Dört taramanın satırları tekil ilanlara nasıl iniyor, veri hangi sınırlarla toplandı ve `ad_id`'nin
göremediği tekrar, yani yeni numarayla yeniden yayımlanmış aynı araç, ne kadar? Sonuncusu bir sızıntı riskidir: aynı
araç çapraz doğrulamada hem eğitim hem test tarafına düşerse tahmin modeli o aracın fiyatını önceden görmüş olur ve
başarısı olduğundan iyi görünür.

#### 1.1.1 Satırlar ve ilanlar

İlk adım basit bir sayım: kaç tarama satırı, kaç ilan, aradaki fark kaç satır.

In [12]:
n_snapshot_rows = len(all_rows)
n_listings = len(listings)
n_repeat_rows = n_snapshot_rows - n_listings
assert all_rows["ad_id"].nunique() == n_listings   # her ilan listings'te tam bir kez

print("Tarama satırı:", tr_int(n_snapshot_rows))
print("İlan:", tr_int(n_listings))
print("Aynı ilanın sonraki görülmeleri:", tr_int(n_repeat_rows))

Tarama satırı: 45.159
İlan: 29.988
Aynı ilanın sonraki görülmeleri: 15.171


In [13]:
listings["brand"].value_counts().rename_axis("marka").rename("ilan").to_frame()

,ilan
marka,
bmw,17896
audi,12092


45.159 satırın 15.171'i bir ilanın sonraki taramalarda yeniden görülmesi. `load_clean` bunlardan yalnız en son
satırı tutuyor. İlanların yaklaşık %60'ı BMW, %40'ı Audi.

#### 1.1.2 Aynı ilan, değişen fiyat

Yeniden görülen bir ilanın fiyatı değişmiş olabilir. Bunu görmek için her ilanın satırlarını tarih sırasına koyup
ilan başına dört şey sayıyoruz: kaç taramada görüldüğü, kaç farklı fiyatı olduğu, ilk ve son fiyatı.

In [14]:
rows_in_order = all_rows.sort_values(["ad_id", "search_date"])
per_ad = rows_in_order.groupby("ad_id").agg(
    n_snapshots=("search_date", "nunique"),
    n_prices=("price", "nunique"),
    first_price=("price", "first"),
    last_price=("price", "last"),
)
print("İlan sayısı:", tr_int(len(per_ad)))

İlan sayısı: 29.988


Bu tablodan dört grup çıkar. Birden çok taramada görülen ilanlardan fiyatı değişenler; bunların içinde son fiyatı
ilkinden düşük olanlar (indirim), yüksek olanlar (zam) ve fiyatı değişip sonunda ilk fiyatına dönenler.

In [15]:
seen_again = per_ad["n_snapshots"] > 1
price_changed = seen_again & (per_ad["n_prices"] > 1)
price_cut = seen_again & (per_ad["last_price"] < per_ad["first_price"])
price_rise = seen_again & (per_ad["last_price"] > per_ad["first_price"])
price_returned = price_changed & ~price_cut & ~price_rise   # değişmiş ama ilk fiyatına dönmüş

price_change_counts = pd.Series({"seen_again": seen_again.sum(), "changed": price_changed.sum(),
                                 "cuts": price_cut.sum(), "rises": price_rise.sum(),
                                 "returned": price_returned.sum()})
price_change_table = pd.DataFrame({"listings": price_change_counts,
                                   "pct_of_all": (100 * price_change_counts / n_listings).round(1)})

price_change_table.rename(
    index={"seen_again": "birden çok taramada görülen", "changed": "fiyatı değişen", "cuts": "indirim",
           "rises": "zam", "returned": "değişip ilk fiyatına dönen"},
    columns={"listings": "ilan", "pct_of_all": "bütün ilanlara oranı (%)"})

,ilan,bütün ilanlara oranı (%)
birden çok taramada görülen,10831,36.1
fiyatı değişen,4703,15.7
indirim,2801,9.3
zam,1854,6.2
değişip ilk fiyatına dönen,48,0.2


Fiyatı değişen ilanlardan birkaçı (ilan numarası olmadan):

In [16]:
changed_examples = per_ad[price_changed].reset_index(drop=True).head(4)

changed_examples.rename(columns={"n_snapshots": "görüldüğü tarama", "n_prices": "farklı fiyat",
                                 "first_price": "ilk fiyat", "last_price": "son fiyat"})

,görüldüğü tarama,farklı fiyat,ilk fiyat,son fiyat
0,2,2,2250000.0,2200000.0
1,3,2,750000.0,850000.0
2,4,2,2150000.0,2125000.0
3,3,2,1150000.0,1125000.0


**Ne görüyoruz.** İlanların üçte biri (10.831, %36.1) birden çok taramada görülmüş; 4.703'ünün (%15.7) fiyatı
değişmiş. İndirim (2.801) zamdan (1.854) daha sık; 48 ilan değişip ilk fiyatına dönmüş. Bu geçmiş bilgi taşıyor ama
tahmin modeli kullanmıyor: her ilanın yalnız son satırı, yani son fiyatı, tahmin modeline giriyor.

#### 1.1.3 Plaka kapsamı

`load_clean` yalnız TR plakalı satırları alıyor. Dışarıda neyin kaldığını görmek için veritabanından, fiyatı 0'dan
büyük bütün satırların iki kolonunu doğrudan okuyoruz: `ad_id` ve plaka. Plaka değerleri sitedeki gibi: TR plakası,
mavi plaka (Türkiye'de oturan yabancıların aracı; vergi rejimi farklı) ya da boş. `DB_PATH` veritabanı dosyasının
yolu, `lib.common`'dan geliyor.

In [17]:
TR_PLATE = "(TR) Türkiye"   # veritabanındaki TR plaka değeri

with duckdb.connect(str(DB_PATH), read_only=True) as con:
    plate_rows = con.execute("SELECT ad_id, gb_plate_origin FROM car_listings WHERE price > 0").df()

plate_rows["plate"] = plate_rows["gb_plate_origin"].fillna("(bilinmiyor)")   # boş plaka; kayıt da bu adı kullanır
plate_rows["is_tr"] = plate_rows["gb_plate_origin"] == TR_PLATE
print("Fiyatı 0'dan büyük satır:", tr_int(len(plate_rows)))

Fiyatı 0'dan büyük satır: 45.315


In [18]:
plate_table = plate_rows.groupby("plate").agg(rows=("ad_id", "size"), listings=("ad_id", "nunique"))
plate_table = plate_table.sort_values("rows", ascending=False)

plate_table.rename_axis("plaka").rename(columns={"rows": "satır", "listings": "ilan"})

,satır,ilan
plaka,,
(TR) Türkiye,45159,29988
(bilinmiyor),118,77
Mavi plakalı,38,22


Bir ilan bir taramada boş plakayla, başka bir taramada TR plakayla görünebilir. Bu yüzden ilan başına iki soru
soruyoruz: hiç TR satırı var mı, satırlarının hepsi TR mi? Hiç TR satırı olmayan ilan eğitime hiç girmez.

In [19]:
ad_plates = plate_rows.groupby("ad_id")["is_tr"].agg(has_tr="any", all_tr="all")
dropped = ~ad_plates["has_tr"]                       # hiç TR satırı yok: dışarıda
mixed = ad_plates["has_tr"] & ~ad_plates["all_tr"]   # hem TR hem başka plakalı satırı var

print("Hiç TR satırı olmayan ilan:", dropped.sum())
print("Hem TR hem başka plakalı satırı olan ilan:", mixed.sum())

Hiç TR satırı olmayan ilan: 76
Hem TR hem başka plakalı satırı olan ilan: 23


Dışarıda kalan ilanlar hangi plakadan? Her birinin tek bir plaka etiketi olmalı; iki farklı etiket görülürse hücre
durur.

In [20]:
dropped_rows = plate_rows[plate_rows["ad_id"].isin(ad_plates.index[dropped])]
assert (dropped_rows.groupby("ad_id")["plate"].nunique() == 1).all(), "iki plaka etiketli ilan"

dropped_by_plate = dropped_rows.drop_duplicates("ad_id")["plate"].value_counts()
dropped_by_plate.rename_axis("plaka").rename("dışarıda kalan ilan").to_frame()

,dışarıda kalan ilan
plaka,
(bilinmiyor),54
Mavi plakalı,22


**Ne görüyoruz.** Fiyatı olan 45.315 satırın 45.159'u TR plakalı. Dışarıda 76 ilan kalıyor: 22 mavi plakalı ve 54
plakası boş ilan. Mavi plakalılar farklı vergi rejimine girdiği için, plakası boş olanlar hangi rejime girdiği
bilinmediği için alınmadı. Plakası bir taramada boş görünen 77 ilanın 23'ü başka bir taramada TR plakalı; bunlar
tahmin modeline TR satırlarıyla giriyor.

#### 1.1.4 Toplama filtreleri

Veri sitenin bütün ilanları değil: veri toplama kodu (`scraper/`) belirli filtrelerle arama yaptı. Filtreler
[scraper/collection_config.json](../../scraper/collection_config.json) dosyasında; toplama kodu da aynı dosyayı
okuyor, burada yalnız okunuyor. Fiyat aralığı marka başına tanımlı; kendi aralığı olmayan bir marka `DEFAULT` aralığını
alır.

In [21]:
config = json.loads((ROOT / "scraper" / "collection_config.json").read_text(encoding="utf-8"))
query = config["listing_query"]
brand_ranges = []
for brand in config["selected_brands"]:
    brand_ranges.append(config["brands"].get(brand, config["brands"]["DEFAULT"]))

filters = {"brands": config["selected_brands"],
           "price_min": min(r["min"] for r in brand_ranges),
           "price_max": max(r["max"] for r in brand_ranges),
           "max_km": query["max_km"], "min_year": query["min_year"],
           "fuel_filter": query["fuels"], "category": query["category"]}

filter_table = pd.Series(filters).to_frame("value")
filter_table.rename(index={"brands": "markalar", "price_min": "en düşük fiyat (₺)", "price_max": "en yüksek fiyat (₺)",
                           "max_km": "en fazla km", "min_year": "en eski model yılı", "fuel_filter": "yakıtlar",
                           "category": "kategori"}, columns={"value": "değer"}).rename_axis("filtre")

,değer
filtre,
markalar,"[audi, bmw]"
en düşük fiyat (₺),300000
en yüksek fiyat (₺),6500000
en fazla km,700000
en eski model yılı,2005
yakıtlar,"[Benzin, Dizel, Hibrit, LPG]"
kategori,otomobil


Filtrelerin verideki izi: veri filtreleri aşıyorsa (ör. tavanın üstünde bir fiyat) ya filtre değişmiştir ya da onu
okuyan kod bozuktur; o durumda hücre durur. Aşmıyorsa sınırlara ne kadar ilanın dayandığını sayıyoruz.

In [22]:
price = listings["price"]
assert filters["price_min"] <= price.min() and price.max() <= filters["price_max"], "fiyat filtrenin dışında"
assert listings["gb_mileage"].max() <= filters["max_km"], "km filtrenin dışında"
assert listings["gb_year"].min() >= filters["min_year"], "model yılı filtrenin dışında"

footprint = pd.Series({
    "price_max": price.max(),
    "at_cap": (price == filters["price_max"]).sum(),
    "within_1pct_of_cap": (price >= 0.99 * filters["price_max"]).sum(),
    "km_max": listings["gb_mileage"].max(),
    "year_min": listings["gb_year"].min(),
    "in_min_year": (listings["gb_year"] == filters["min_year"]).sum(),
    "suv": (listings["kb_body_type"] == "SUV").sum(),
    "electric": listings["kb_fuel"].str.contains("Elektrik").sum(),
}, dtype=object)   # object: tam sayılar tam sayı olarak kalsın
max_age = listings["search_date"].max().year - filters["min_year"]   # en yaşlı aracın olabileceği yaş

In [23]:
print("Olabilecek en büyük yaş:", max_age)
footprint_table = footprint.to_frame("in_data")
footprint_table.rename(index={"price_max": "en yüksek fiyat (₺)", "at_cap": "tam tavanda (ilan)",
                              "within_1pct_of_cap": "tavanın %1 yakınında (ilan)", "km_max": "en yüksek km",
                              "year_min": "en eski model yılı", "in_min_year": "en eski model yılında (ilan)",
                              "suv": "SUV (ilan)", "electric": "elektrikli (ilan)"}, columns={"in_data": "veride"})

Olabilecek en büyük yaş: 21


,veride
en yüksek fiyat (₺),6500000.0
tam tavanda (ilan),10
tavanın %1 yakınında (ilan),38
en yüksek km,648000
en eski model yılı,2005
en eski model yılında (ilan),541
SUV (ilan),6
elektrikli (ilan),0


In [24]:
listings["kb_fuel"].value_counts().rename_axis("yakıt").rename("ilan").to_frame()

,ilan
yakıt,
Benzin,14882
Dizel,12982
LPG & Benzin,1224
Hibrit,900


**Ne görüyoruz.** Üç sonuç var.

- **Fiyat sağdan kesik.** En pahalı ilan tam tavanda (6.500.000 ₺); 10 ilan tavanda, 38 ilan tavanın %1 yakınında,
  üstünde hiç yok. Tavanın üstündeki araçlar veride değil; en pahalı uçtaki tahminler bu sınırla birlikte okunmalı.
- **Yaş en fazla 21.** 2005 model yılında 541 ilan var; daha eski araçlar toplanmadı. En yaşlı yaş grubu doğal bir
  uç değil, toplama sınırı.
- **Gövde ve yakıt.** Yalnız otomobil kategorisi toplandı: veride yalnız 6 SUV var. Yakıt filtresi elektrikliyi
  dışarıda bırakıyor: elektrikli ilan yok. "LPG" filtresi veride "LPG & Benzin" olarak görünüyor.

#### 1.1.5 İçerik tekrarı: `ad_id`'nin göremediği aynı araç

Satıcı aynı aracı ilanı kaldırıp yeni bir numarayla yeniden yayımlayabilir. Bu iki ilanın `ad_id`'si farklıdır;
`ad_id` başına tekilleştirme onları ayrı ilan sayar. Çapraz doğrulama birini eğitime, ötekini teste koyarsa sızıntı
olur. Bunu kesin bilemeyiz, ama bütün ayırt edici kolonları aynı olan ilanları sayabiliriz. Üç tanım:

- **katı:** fiyat, km, model yılı, marka, seri, araç modeli, yakıt, ağır hasar, boyalı ve değişen parça sayısı
  (veritabanındaki `count_painted`, `count_changed`), güç ve hacmin üst sınırı.
- **gevşek:** yalnız fiyat, km, model yılı ve araç modeli.
- **fiyat hariç katı:** katı tanımdan fiyat çıkarılmış. Yeni fiyatla yeniden yayımlanan ilan yalnız burada
  görünür.

Ölçü "fazla satır": aynı değerleri taşıyan her grupta ilk ilandan sonraki ilanlar. pandas'ın `duplicated` metodu
tam bunu işaretler: verilen kolonlardaki değerleri daha önce görülmüş her satır. Boş değerler birbirine eşit sayılır.

In [25]:
STRICT_KEY = ["price", "gb_mileage", "gb_year", "brand", "series", "model", "kb_fuel",
              "is_heavy_damaged", "count_painted", "count_changed", "power_hp_up", "engine_cc_up"]
LOOSE_KEY = ["price", "gb_mileage", "gb_year", "model"]
NO_PRICE_KEY = [column for column in STRICT_KEY if column != "price"]

strict_extra = listings.duplicated(subset=STRICT_KEY).sum()
loose_extra = listings.duplicated(subset=LOOSE_KEY).sum()
no_price_extra = listings.duplicated(subset=NO_PRICE_KEY).sum()

duplicate_table = pd.DataFrame({"extra_rows": [strict_extra, loose_extra, no_price_extra]},
                               index=["strict", "loose", "no_price"])
duplicate_table["pct"] = (100 * duplicate_table["extra_rows"] / n_listings).round(2)

duplicate_table.rename(index={"strict": "katı", "loose": "gevşek", "no_price": "fiyat hariç katı"},
                       columns={"extra_rows": "fazla satır", "pct": "ilanlara oranı (%)"}).rename_axis("tanım")

,fazla satır,ilanlara oranı (%)
tanım,,
katı,137,0.46
gevşek,209,0.70
fiyat hariç katı,1278,4.26


Fiyat hariç tanımda bir sorun var: kilometre çoğunlukla yuvarlak yazılıyor (180.000 gibi), bu yüzden aynı araç
modelinin iki farklı aracı km'de kolayca çakışabilir. Kilometresi yuvarlak olmayan ilanlarda böyle bir tesadüf çok
daha zor; aynı sayım orada da yapılıyor.

In [26]:
round_km = listings["gb_mileage"] % 1000 == 0
no_price_non_round_extra = listings[~round_km].duplicated(subset=NO_PRICE_KEY).sum()

print(f"Kilometresi 1.000'in katı olan ilan: %{100 * round_km.mean():.1f}")
print("Yuvarlak olmayan km'de fiyat hariç katı fazla satır:", no_price_non_round_extra,
      f"(bu ilanların %{100 * no_price_non_round_extra / (~round_km).sum():.1f}'i)")

Kilometresi 1.000'in katı olan ilan: %86.5
Yuvarlak olmayan km'de fiyat hariç katı fazla satır: 113 (bu ilanların %2.8'i)


Katı tanımın grupları: kaç grup var ve en çok tekrar edenler hangileri? Grup başına satır sayısı `groupby(...).size()`
ile bulunuyor; fazla satır, her grubun boyutundan bir eksiğinin toplamı. Bu, `duplicated` ile bulunan sayıyla aynı
olmalı.

In [27]:
strict_group_sizes = listings.groupby(STRICT_KEY, dropna=False).size()
repeated_groups = strict_group_sizes[strict_group_sizes > 1]
assert (repeated_groups - 1).sum() == strict_extra
print("Katı tanımda tekrar grubu:", len(repeated_groups))

most_repeated = repeated_groups.nlargest(5).reset_index(name="n")
most_repeated[["model", "gb_year", "price", "n"]].rename(
    columns={"model": "araç modeli", "gb_year": "model yılı", "price": "fiyat (₺)", "n": "ilan"})

Katı tanımda tekrar grubu: 127


,araç modeli,model yılı,fiyat (₺),ilan
0,320i Sport Line,2025,4900000.0,4
1,318i Standart,2005,717000.0,3
2,A3 Sportback 1.6 Ambition,2010,890000.0,3
3,116d Joy Plus,2015,1044950.0,3
4,320i First Edition Sport Line,2019,2530000.0,3


**Ne görüyoruz.** Katı tanım 127 grupta 137 fazla satır buluyor (%0.46); gevşek tanım 209 (%0.70). Bunların bir
kısmı gerçek tekrar ilan, bir kısmı yaygın araç modellerinde tesadüfi çakışma. İlk iki tanım fiyat eşitliği istediği
için fiyatı değiştirilip yeniden yayımlanan ilanı göremez; fiyat hariç tanım 1.278 fazla satır (%4.26) buluyor.
İlanların %86.5'inde kilometre yuvarlak, bu yüzden bu sayının bir kısmı tesadüfi çakışma olabilir. Tesadüfün zor
olduğu yuvarlak olmayan km'de de 113 fazla satır kalıyor: aynı aracın yeniden yayımlanması gerçek bir olgu. Ama
1.278'in ne kadarının fiyatı değiştirilmiş yeniden ilan, ne kadarının tesadüf olduğu veriden ayrılamıyor.

Çapraz doğrulamada katlar arasına sızabilecek pay bu tanımlara göre %0.46 (katı) ile %4.26 (fiyat hariç) arasında.
Kilometresi de değiştirilmiş bir yeniden ilanı hiçbir tanım göremez. Bu tekrarların tahmin modelinin başarısına
etkisi ölçülmedi (betiğin kartında yazılı).

#### 1.1.6 Kayıtla karşılaştırma

Yukarıda hesaplanan sayılar `metrics/01_dedup_leakage.json` kaydıyla aynı mı?

In [28]:
record = read_record("01_dedup_leakage")
meta = record["meta"]
changes = record["error_drivers"]["price_changes"]
plates = record["error_drivers"]["plate_scope"]
scope = record["error_drivers"]["scope"]
duplicates = record["methodology"]["content_duplicates"]

# Satırlar, ilanlar, taramalar, markalar, fiyat değişimleri
assert meta["n_raw"] == n_snapshot_rows and meta["n_dedup"] == n_listings
assert meta["snapshots"] == sorted(listings["snap"].unique())
assert meta["brands"] == listings["brand"].value_counts().to_dict()
for name, count in price_change_counts.items():
    assert changes[name] == count, name
# Plaka
for item in plates["distribution"]:
    assert plate_table.loc[item["plate"]].tolist() == [item["rows"], item["listings"]], item["plate"]
assert plates["in_training"] == plate_table.loc[TR_PLATE, "listings"]
assert plates["dropped_listings"] == dropped.sum() and plates["listings_with_both"] == mixed.sum()
assert len(plates["dropped_by_plate"]) == len(dropped_by_plate)
for item in plates["dropped_by_plate"]:
    assert dropped_by_plate[item["plate"]] == item["listings"], item["plate"]
# Toplama filtreleri ve verideki izleri
for name, value in filters.items():
    assert scope[name] == value, name
for name, value in footprint.items():
    assert scope["measured"][name] == value, name
assert scope["measured"]["fuel"] == listings["kb_fuel"].value_counts().to_dict()
assert scope["max_age"] == max_age
# İçerik tekrarı
assert duplicates["strict_key_columns"] == STRICT_KEY and duplicates["n_duplicate_groups"] == len(repeated_groups)
assert duplicates["strict_extra"] == strict_extra and duplicates["loose_extra"] == loose_extra
assert duplicates["no_price_extra"] == no_price_extra
assert duplicates["no_price_non_round_km_extra"] == no_price_non_round_extra
assert duplicates["round_km_pct"] == round(100 * round_km.mean(), 1)
print(f"Kayıtla aynı: {tr_int(n_snapshot_rows)} satır, {tr_int(n_listings)} ilan; "
      f"{tr_int(price_change_counts['changed'])} fiyat değişimi; {dropped.sum()} ilan plakası yüzünden dışarıda; "
      f"içerik tekrarı katı / gevşek / fiyat hariç {strict_extra} / {loose_extra} / {tr_int(no_price_extra)}.")

Kayıtla aynı: 45.159 satır, 29.988 ilan; 4.703 fiyat değişimi; 76 ilan plakası yüzünden dışarıda; içerik tekrarı katı / gevşek / fiyat hariç 137 / 209 / 1.278.


### 1.2 Motor gücü ve hacmi: aralıktan tek sayıya

Kaynak: [betik](../../analysis/01_engine_rule.py) · [kart](../../analysis/cards/01_engine_rule.json) ·
[kayıt](../../metrics/01_engine_rule.json)

**Soru.** Site motor hacmini ve gücünü ilanların bir kısmında kesin değer, bir kısmında aralık olarak veriyor (ör.
1401-1600 cc). Tahmin modeli tek sayı istiyor. Aralığın alt sınırı mı, orta noktası mı, üst sınırı mı kullanılmalı?
Projede seçilen kural: **hacim = üst sınır, güç = iki sınırın ortalaması** (orta nokta). Burada bu kural veriyle
sınanıyor. Fiyat hiç kullanılmıyor; kural fiyattan bilgi taşıyamaz.

#### 1.2.1 Kural veride

Önce kuralın `load_clean`'in verdiği veride gerçekten uygulandığını görelim. Aşağıdaki hücre her iki eşitliği bütün
ilanlarda sınar (boş değerler aynı yerde boş olmalı).

In [29]:
assert listings["engine_cc_val"].equals(listings["engine_cc_up"])
assert listings["power_hp_val"].equals(listings[["power_hp_low", "power_hp_up"]].mean(axis=1))
print("Kural bütün ilanlarda geçerli: hacim = üst sınır, güç = iki sınırın ortalaması.")

Kural bütün ilanlarda geçerli: hacim = üst sınır, güç = iki sınırın ortalaması.

Aralıklı ilanlardan örnekler: sitenin verdiği iki sınır ve tahmin modelinin kullandığı tek sayı.

In [30]:
engine_columns = ["model", "engine_cc_low", "engine_cc_up", "engine_cc_val",
                  "power_hp_low", "power_hp_up", "power_hp_val"]
has_cc_range = listings["engine_cc_is_range"].fillna(False)
range_examples = listings.loc[has_cc_range, engine_columns].head(4)

range_examples.rename(columns={"model": "araç modeli", "engine_cc_low": "hacim alt", "engine_cc_up": "hacim üst",
                               "engine_cc_val": "hacim (kullanılan)", "power_hp_low": "güç alt",
                               "power_hp_up": "güç üst", "power_hp_val": "güç (kullanılan)"})

,araç modeli,hacim alt,hacim üst,hacim (kullanılan),güç alt,güç üst,güç (kullanılan)
1,730d Standart,2501.0,3000.0,3000.0,226.0,250.0,238.0
19,116i Comfort,1401.0,1600.0,1600.0,126.0,150.0,138.0
37,525d Standart,2501.0,3000.0,3000.0,251.0,275.0,263.0
47,A4 Sedan 2.0 TDI,1801.0,2000.0,2000.0,126.0,150.0,138.0


#### 1.2.2 Kesin, aralıklı, boş

Hesap hacim ve güç için aynı. Kodu iki kez yazmamak için iki ölçüyü alt alta tek bir tabloda topluyoruz: her ilan
iki satır olur, biri hacim (`engine_cc`), biri güç (`power_hp`). Her satırda araç modeli, sitenin verdiği alt ve üst
sınır ve "aralık mı" bayrağı var. Kesin değerde alt ve üst sınır aynı sayıdır.

In [31]:
MEASURES = ["engine_cc", "power_hp"]
MEASURE_TR = {"engine_cc": "hacim (cc)", "power_hp": "güç (hp)"}   # yalnız gösterim için

bound_parts = []
for measure in MEASURES:
    part = pd.DataFrame({"measure": measure,
                         "model": listings["model"],
                         "low": listings[f"{measure}_low"],
                         "up": listings[f"{measure}_up"],
                         "is_range": listings[f"{measure}_is_range"].fillna(False).astype(bool)})
    bound_parts.append(part)
bounds = pd.concat(bound_parts, ignore_index=True)
print("Satır:", tr_int(len(bounds)), "(her ilan iki kez)")

Satır: 59.976 (her ilan iki kez)


Her satır üç türden birine girer: **kesin** (aralık değil ve değeri var), **aralıklı** (bazıları açık uçlu: yalnız
bir sınırı var, ör. "1200 cc'ye kadar") ya da **değer yok**.

In [32]:
bounds["is_exact"] = ~bounds["is_range"] & bounds["up"].notna()
bounds["is_open_ended"] = bounds["is_range"] & (bounds["low"].isna() != bounds["up"].isna())
bounds["has_no_value"] = bounds["low"].isna() & bounds["up"].isna()

split_counts = bounds.groupby("measure")[["is_exact", "is_range", "is_open_ended", "has_no_value"]].sum()

split_counts.rename(index=MEASURE_TR, columns={"is_exact": "kesin", "is_range": "aralıklı",
                                               "is_open_ended": "aralıklı: açık uçlu",
                                               "has_no_value": "değer yok"}).rename_axis("ölçü")

,kesin,aralıklı,aralıklı: açık uçlu,değer yok
ölçü,,,,
hacim (cc),21767,7865,38,356
güç (hp),21765,7797,2,426


**Ne görüyoruz.** Her iki ölçüde ilanların yaklaşık dörtte biri aralıklı: hacimde 7.865, güçte 7.797 ilan. Açık
uçlu aralık az (hacimde 38, güçte 2); orada tahmin modeli bilinen tek sınırı kullanıyor. Hacmi boş 356, gücü boş 426
ilan var.

#### 1.2.3 Referans: aynı araç modelinin kesin değeri

Aralıklı bir ilanın gerçek değerini bilmiyoruz. En iyi vekili, aynı araç modelinin değeri kesin verilmiş ilanları:
onların medyanı o araç modelinin **referansı**. Karşılaştırmaya yalnız iki sınırı da bilinen ve araç modelinin en az
bir kesin değerli ilanı olan aralıklı ilanlar girer.

In [33]:
exact = bounds[bounds["is_exact"]]
reference = exact.groupby(["measure", "model"])["up"].median().rename("reference").reset_index()

ranged = bounds[bounds["is_range"] & bounds["low"].notna() & bounds["up"].notna()]
ranged = ranged.merge(reference, on=["measure", "model"], how="left")
ranged = ranged[ranged["reference"].notna()].copy()

reference_counts = ranged.groupby("measure").agg(listings=("model", "size"), models=("model", "nunique"))

reference_counts.rename(index=MEASURE_TR, columns={"listings": "referanslı aralıklı ilan",
                                                   "models": "araç modeli"}).rename_axis("ölçü")

,referanslı aralıklı ilan,araç modeli
ölçü,,
hacim (cc),7176,339
güç (hp),7148,337


#### 1.2.4 Üç aday, üç uzaklık

Her aralıklı ilan için üç aday var: alt sınır, orta nokta ve üst sınır. Her adayın referanstan mutlak uzaklığını
hesaplayıp ölçü başına medyanını alıyoruz (tablodaki sayılar cc ve hp cinsinden medyan mutlak uzaklık). Medyan
uzaklığı en küçük aday, tek kural olarak en iyisi.

In [34]:
CANDIDATES = ["lower", "mid", "upper"]
CANDIDATE_TR = {"lower": "alt sınır", "mid": "orta nokta", "upper": "üst sınır"}   # yalnız gösterim için

ranged["lower"] = (ranged["low"] - ranged["reference"]).abs()
ranged["mid"] = ((ranged["low"] + ranged["up"]) / 2 - ranged["reference"]).abs()
ranged["upper"] = (ranged["up"] - ranged["reference"]).abs()

median_gaps = ranged.groupby("measure")[CANDIDATES].median()
median_gaps.rename(index=MEASURE_TR, columns=CANDIDATE_TR).rename_axis("ölçü")

,alt sınır,orta nokta,üst sınır
ölçü,,,
hacim (cc),194.0,95.5,5.0
güç (hp),17.0,7.0,14.0


Kural, seçilen adayın öteki iki adayın her birinden daha yakın olmasını ister; eşitlik yetmez. Değilse kural
veriyle çelişiyordur ve hücre durur.

In [35]:
CHOSEN = {"engine_cc": "upper", "power_hp": "mid"}   # projenin kuralı

for measure, chosen in CHOSEN.items():
    others = median_gaps.loc[measure].drop(chosen)
    assert (median_gaps.loc[measure, chosen] < others).all(), f"{measure}: seçilen aday en yakın değil"
    print(f"{MEASURE_TR[measure]}: seçilen aday ({CANDIDATE_TR[chosen]}) en yakın.")

hacim (cc): seçilen aday (üst sınır) en yakın.
güç (hp): seçilen aday (orta nokta) en yakın.


**Ne görüyoruz.** Hacimde üst sınır referansa medyan 5 cc uzak; orta nokta 95.5 cc, alt sınır 194 cc. Güçte orta
nokta 7 hp ile en yakın; üst sınır 14 hp, alt sınır 17 hp. Kural iki ölçüde de veriyle uyumlu.

Raporda aynı karşılaştırma dağılımıyla birlikte çiziliyor. Aşağıdaki görsel raporun Figür 29'u: burada
hesaplanmadı, raporu üreten kod kayıttan çiziyor; kutular çeyrekleri, bıyıklar %5 ve %95 yüzdeliklerini gösteriyor.

![Aralıktan tek sayıya: adayın aynı araç modelinin kesin değerinden uzaklığı](../../reports/figures/tr-29-hp-cc-rule.png)

#### 1.2.5 Referans aralığın neresinde?

Uzaklığın arkasındaki yapı: referans kendi aralığının neresine düşüyor? **Konum** = (referans - alt sınır) / (üst
sınır - alt sınır); 0 alt sınır, 1 üst sınır demek. Referans aralığın dışındaysa konum 0'ın altında ya da 1'in
üstünde olur. Ayrıca referansın aralığın içinde kaldığı ilanların payını sayıyoruz.

In [36]:
width = ranged["up"] - ranged["low"]
ranged["position"] = (ranged["reference"] - ranged["low"]) / width.where(width > 0)   # genişliği 0 olan aralık yok sayılır
ranged["inside"] = (ranged["reference"] >= ranged["low"]) & (ranged["reference"] <= ranged["up"])

inside_share = ranged.groupby("measure")["inside"].mean()
position_quartiles = ranged.groupby("measure")["position"].quantile([0.25, 0.5, 0.75]).unstack()

position_table = position_quartiles.round(3)
position_table["inside_pct"] = (100 * inside_share).round(1)

position_table.rename(index=MEASURE_TR, columns={0.25: "konum: alt çeyrek", 0.5: "konum: medyan",
                                                 0.75: "konum: üst çeyrek",
                                                 "inside_pct": "aralığın içinde (%)"}).rename_axis("ölçü")

,konum: alt çeyrek,konum: medyan,konum: üst çeyrek,aralığın içinde (%)
ölçü,,,,
hacim (cc),0.839,0.975,0.990,97.9
güç (hp),0.292,0.583,0.792,84.2


**Ne görüyoruz.** Hacimde referans neredeyse hep aralığın üst ucunda: medyan konum 0.975, alt çeyrek bile 0.839;
ilanların %97.9'unda referans aralığın içinde. Sebep motorların litre etiketi: "1.6" diye satılan bir motorun gerçek
hacmi 1598 cc gibi, etiketin birkaç cc altında. Sitenin aralıkları da bu etiket değerlerinde bitiyor (1401-1600 cc),
bu yüzden gerçek hacim neredeyse her zaman üst sınırın dibinde. Güçte böyle bir etiket yok: konum dağınık (çeyrekler
0.292 ile 0.792 arası) ve referans ilanların %84.2'sinde aralığın içinde. Dağınık bir konumda ortalama, yani orta
nokta, en iyi tutan tek sayı.

#### 1.2.6 Aralık aralık en yakın aday

Tek kural bütün aralıklı ilanlara bakıyor. Aralık bazında da aynı mı? En az 100 ilanı olan her aralık için üç
adayın medyan uzaklığını hesaplayıp en yakını seçiyoruz. İki aday berabere kalırsa ikisi de yazılır (tabloda ör.
"alt+orta").

In [37]:
MIN_BUCKET_N = 100

bucket_medians = ranged.groupby(["measure", "low", "up"]).agg(
    n=("lower", "size"), lower=("lower", "median"), mid=("mid", "median"), upper=("upper", "median"))
big_buckets = bucket_medians[bucket_medians["n"] >= MIN_BUCKET_N].copy()

closest = []
for _, row in big_buckets.iterrows():
    best = row[CANDIDATES].min()
    winners = [name for name in CANDIDATES if row[name] == best]
    closest.append("+".join(winners))
big_buckets["closest"] = closest

In [38]:
shown = big_buckets.reset_index()
shown["measure"] = shown["measure"].map(MEASURE_TR)
shown["closest"] = shown["closest"].str.replace("lower", "alt").str.replace("mid", "orta").str.replace("upper", "üst")

shown.rename(columns={"measure": "ölçü", "low": "alt sınır", "up": "üst sınır", "n": "ilan",
                      "lower": "uzaklık: alt", "mid": "uzaklık: orta", "upper": "uzaklık: üst",
                      "closest": "en yakın"})

,ölçü,alt sınır,üst sınır,ilan,uzaklık: alt,uzaklık: orta,uzaklık: üst,en yakın
0,hacim (cc),1201.0,1400.0,123,189.0,89.5,10.0,üst
1,hacim (cc),1401.0,1600.0,4015,196.0,96.5,3.0,üst
2,hacim (cc),1601.0,1800.0,146,197.0,97.5,2.0,üst
3,hacim (cc),1801.0,2000.0,2572,167.0,67.5,32.0,üst
4,hacim (cc),2501.0,3000.0,248,466.0,216.5,33.0,üst
5,güç (hp),101.0,125.0,1266,9.0,3.0,15.0,orta
6,güç (hp),126.0,150.0,1361,24.0,12.0,0.0,üst
7,güç (hp),151.0,175.0,2128,19.0,7.0,5.0,üst
8,güç (hp),176.0,200.0,1695,8.0,11.0,23.0,alt
9,güç (hp),201.0,225.0,256,3.0,9.0,21.0,alt


**Ne görüyoruz.** Hacimde en az 100 ilanlı beş aralığın hepsinde en yakın aday üst sınır. Güçte en yakın aday güç
düzeyine göre değişiyor: 101-125 hp'de orta nokta, 126-175 hp'de üst sınır, 176-225 hp'de alt sınır, 226-250 hp'de
alt sınır ile orta nokta berabere, 251-275 hp'de alt sınır. Hiçbiri her yerde kazanmıyor; tek bir kural olarak orta
nokta, bütün aralıklı ilanlarda medyan uzaklığı en küçük aday.

#### 1.2.7 Bir örnek ve aralığın dışına düşen referans

Somut bir örnek: hacimde en sık görülen aralık hangisi, o aralıktaki araç modellerinin kesin değerli ilanlarında en
sık hangi hacim yazıyor?

In [39]:
cc_ranged = ranged[ranged["measure"] == "engine_cc"]
cc_bucket_sizes = cc_ranged.groupby(["low", "up"]).size().sort_values(ascending=False)
top_low, top_up = cc_bucket_sizes.index[0]

in_top_bucket = (cc_ranged["low"] == top_low) & (cc_ranged["up"] == top_up)
models_in_top_bucket = cc_ranged.loc[in_top_bucket, "model"].unique()
cc_exact = exact[exact["measure"] == "engine_cc"]
exact_values = cc_exact.loc[cc_exact["model"].isin(models_in_top_bucket), "up"].value_counts()

print(f"En sık hacim aralığı: {top_low:.0f}-{top_up:.0f} cc ({tr_int(cc_bucket_sizes.iloc[0])} ilan)")
print(f"Bu araç modellerinde en sık kesin hacim: {exact_values.index[0]:.0f} cc ({tr_int(exact_values.iloc[0])} ilan)")

En sık hacim aralığı: 1401-1600 cc (4.015 ilan)
Bu araç modellerinde en sık kesin hacim: 1598 cc (4.279 ilan)


Referans aralığın dışına düştüğünde suçlu kim: sitenin aralığı mı, referans mı? Bir araç modeli adı birden çok motor
seçeneğini kapsayabilir; o zaman kesin değerli ilanları birden çok farklı değer taşır ve medyanları kaba bir
referanstır. Aralığın dışına düşen ilanlarda, araç modelinin birden çok kesin değeri olanların payı:

In [40]:
n_exact_values = exact.groupby(["measure", "model"])["up"].nunique().rename("n_exact_values").reset_index()
outside = ranged[~ranged["inside"]].merge(n_exact_values, on=["measure", "model"], how="left")
outside_multi_share = (outside["n_exact_values"] > 1).groupby(outside["measure"]).mean()

outside_table = (100 * outside_multi_share).round(1).to_frame("multi_value_pct")
outside_table.rename(index=MEASURE_TR, columns={"multi_value_pct": "birden çok kesin değeri olan araç modeli (%)"}
                     ).rename_axis("ölçü")

,birden çok kesin değeri olan araç modeli (%)
ölçü,
hacim (cc),26.4
güç (hp),90.3


**Ne görüyoruz.** En sık hacim aralığı 1401-1600 cc (referanslı 4.015 ilan); bu aralıktaki araç modellerinin kesin
değerli ilanlarında en sık hacim 1598 cc. Yani bu ilanlar için alt sınır 1401 (197 cc uzak), orta nokta 1500.5 (97.5
cc), üst sınır 1600 (2 cc). Güçte referansın aralık dışında kaldığı ilanların %90.3'ünde araç modelinin birden çok
kesin güç değeri var: aynı araç modeli adı farklı motor seçenekleri taşıyor. Yani güçteki %15.8'lik "dışarıda" payı
sitenin aralığının değil, referansın kabalığını gösteriyor.

#### 1.2.8 Kayıtla karşılaştırma

In [41]:
record = read_record("01_engine_rule")["error_drivers"]["engine_rule"]

for measure in MEASURES:
    recorded = record[measure]
    assert recorded["chosen"] == CHOSEN[measure] and recorded["listings"] == n_listings
    assert recorded["exact"] == split_counts.loc[measure, "is_exact"]
    assert recorded["ranged"] == split_counts.loc[measure, "is_range"]
    assert recorded["open_ended"] == split_counts.loc[measure, "is_open_ended"]
    assert recorded["no_value"] == split_counts.loc[measure, "has_no_value"]
    assert recorded["with_reference"] == reference_counts.loc[measure, "listings"]
    assert recorded["reference_models"] == reference_counts.loc[measure, "models"]
    for candidate in CANDIDATES:
        assert recorded["candidates"][candidate]["median"] == round(median_gaps.loc[measure, candidate], 1)
    assert recorded["inside_pct"] == round(100 * inside_share[measure], 1)
    assert recorded["position_median"] == round(position_quartiles.loc[measure, 0.5], 3)
    assert recorded["position_quartiles"] == [round(position_quartiles.loc[measure, 0.25], 3),
                                              round(position_quartiles.loc[measure, 0.75], 3)]
    assert recorded["outside_multi_value_pct"] == round(100 * outside_multi_share[measure], 1)
    recorded_winners = []
    for low, up, n, winner, _ in recorded["range_winners"]:
        recorded_winners.append((low, up, n, winner))
    computed_winners = []
    for (low, up), row in big_buckets.loc[measure].iterrows():
        computed_winners.append((low, up, row["n"], row["closest"]))
    assert recorded_winners == computed_winners, measure

example = record["engine_cc"]["example_range"]
assert (example["low"], example["up"], example["n"]) == (top_low, top_up, cc_bucket_sizes.iloc[0])
assert (example["most_common_exact"], example["most_common_exact_n"]) == (exact_values.index[0], exact_values.iloc[0])
print("Kayıtla aynı: kesin / aralıklı / boş sayıları, üç adayın medyan uzaklığı, konum, aralık bazında "
      "kazananlar ve 1401-1600 cc örneği, iki ölçüde de.")

Kayıtla aynı: kesin / aralıklı / boş sayıları, üç adayın medyan uzaklığı, konum, aralık bazında kazananlar ve 1401-1600 cc örneği, iki ölçüde de.


### 1.3 "Belirtilmemiş" hasar cevabı

Kaynak: [betik](../../analysis/01_unspecified_panels.py) · [kart](../../analysis/cards/01_unspecified_panels.json) ·
[kayıt](../../metrics/01_unspecified_panels.json)

**Soru.** Site her ilan için 13 kaporta parçasının her birine beş cevaptan birini veriyor: orijinal, belirtilmemiş,
boyalı, lokal boyalı, değişmiş. Veritabanı her parçayı üç bayrakla tutar (değişen, boyalı, lokal boyalı);
"belirtilmemiş" cevabında üç bayrak da boş kalır. `load_clean` boş bayrağı 0 sayar (Bölüm 0.2'deki kapı örneğinde
`<NA>`), yani belirtilmemiş parça tahmin modeline orijinal görünür. Bu bilinçli bir karar: satıcı hasarı yazmayı
unutmuş olabilir, ama hasar olmaması daha olası sayıldı. Kararın kapsamı ne: kaç parça ve kaç ilan belirtilmemiş
cevap taşıyor?

Betik sayımı sitenin ham cevaplarından yapıyor. Önce ham cevapların tahmin modelinin gördüğü veriyle bire bir aynı
olduğu sınanıyor, sonra sayılıyor.

#### 1.3.1 Sitenin cevapları ve veritabanının kodları

Ham cevabı koda çeviren eşleme [db/lib/damage_mappings.json](../../db/lib/damage_mappings.json) dosyasında;
veritabanını kuran kod da aynı dosyayı okuyor. İki sözlük var: parça adı → durum kolonu ve cevap → kod.

In [42]:
damage_maps = json.loads((ROOT / "db" / "lib" / "damage_mappings.json").read_text(encoding="utf-8"))
panel_map = damage_maps["panels"]      # sitedeki parça adı -> durum kolonu (ör. "Tavan" -> "roof_status")
status_map = damage_maps["statuses"]   # sitedeki cevap -> kod (ör. "Belirtilmemiş" -> "unspecified")
STATUS_COLUMNS = list(panel_map.values())

print(len(panel_map), "parça:", ", ".join(panel_map))
pd.Series(status_map).rename_axis("sitedeki cevap").rename("kod").to_frame()

13 parça: Tavan, Motor Kaputu, Arka Kaput, Ön Tampon, Arka Tampon, Sol Ön Kapı, Sağ Ön Kapı, Sol Arka Kapı, Sağ Arka Kapı, Sol Ön Çamurluk, Sağ Ön Çamurluk, Sol Arka Çamurluk, Sağ Arka Çamurluk


,kod
sitedeki cevap,
Orjinal,original
Belirtilmemiş,unspecified
Boyalı,painted
Lokal boyalı,local_painted
Değişmiş,changed


#### 1.3.2 Ham dosyalardan hasar listeleri

Veri toplama kodunun kaydettiği ham sayfalar `data/raw/<marka>/<tarama>/details.jsonl` dosyalarında: her satır bir ilan
sayfasının bir taramadaki hâli, JSON olarak. Her sayfadan yalnız üç alan okunuyor: ilan numarası (`KısaBilgi - İlan
No`, yalnız rakamları), tarama tarihi (`search_date`, ilk 10 karakteri gün) ve hasar listesi (`Hasar_Listesi`). Öteki
alanlara, satıcı açıklaması dahil, dokunulmuyor; dosyalar yalnız okunuyor.

In [43]:
raw_records = []
n_pages = 0
for path in sorted((ROOT / "data" / "raw").glob("*/*/details.jsonl")):
    with open(path, encoding="utf-8") as file:
        for line in file:
            page = json.loads(line)
            n_pages += 1
            ad_number = re.sub(r"\D", "", str(page.get("KısaBilgi - İlan No") or ""))   # yalnız rakamlar
            if ad_number and page.get("Hasar_Listesi"):
                raw_records.append((int(ad_number), str(page["search_date"])[:10], page["Hasar_Listesi"]))

print("Okunan sayfa:", tr_int(n_pages), "· hasar listesi olan:", tr_int(len(raw_records)))

Okunan sayfa: 45.335 · hasar listesi olan: 45.315


Bir hasar listesi böyle görünüyor: 13 metin, her biri "parça: cevap".

In [44]:
raw_records[0][2]

['Sağ Arka Çamurluk: Orjinal',
 'Arka Kaput: Orjinal',
 'Sol Arka Çamurluk: Orjinal',
 'Sağ Arka Kapı: Orjinal',
 'Sağ Ön Kapı: Orjinal',
 'Tavan: Orjinal',
 'Sol Arka Kapı: Orjinal',
 'Sol Ön Kapı: Orjinal',
 'Sol Ön Çamurluk: Orjinal',
 'Ön Tampon: Orjinal',
 'Arka Tampon: Orjinal',
 'Sağ Ön Çamurluk: Boyalı',
 'Motor Kaputu: Değişmiş']

#### 1.3.3 Listeden tabloya

Her listeyi bir satıra çeviriyoruz: ilan numarası, tarama günü ve 13 parçanın kodu. Parça adı ve cevap eşlemeden
geçiyor; eşlemede olmayan bir parça ya da cevap görülürse (veride hiç görülmedi) hücre hata verip durur, sessizce
atlamaz.

In [45]:
damage_rows = []
for ad_id, snapshot_day, damage_list in raw_records:
    row = {"ad_id": ad_id, "snap": snapshot_day}
    for entry in damage_list:
        panel, answer = entry.split(":", 1)
        row[panel_map[panel.strip()]] = status_map[answer.strip()]
    damage_rows.append(row)
damage = pd.DataFrame(damage_rows)
print("Tablo (satır, kolon):", damage.shape)

Tablo (satır, kolon): (45315, 15)


Aynı ilan aynı taramada iki kez görüldüyse hangisinin doğru olduğu bilinemez; ikisi de atılır. Ardından tablonun ilk
üç satırı, ilan numarası olmadan; tablo yan çevrildi (`.T`), yani her kolon bir sayfa.

In [46]:
duplicated_pages = damage.duplicated(["ad_id", "snap"], keep=False)
damage = damage[~duplicated_pages]
print("Aynı taramada iki kez görülen sayfa:", duplicated_pages.sum())

damage.drop(columns="ad_id").head(3).T

Aynı taramada iki kez görülen sayfa: 0


,0,1,2
snap,2026-01-18,2026-01-18,2026-01-18
fender_rr_status,original,unspecified,original
trunk_lid_status,original,unspecified,original
fender_rl_status,original,unspecified,original
door_rr_status,original,unspecified,original
door_fr_status,original,unspecified,changed
roof_status,original,unspecified,original
door_rl_status,original,unspecified,original
door_fl_status,original,unspecified,original
fender_fl_status,original,unspecified,painted


#### 1.3.4 İlanlarla eşleştirme ve sınama

Ham cevaplar ilanlara (`listings`) ilan numarası ve tarama günüyle bağlanıyor: her ilan, son görüldüğü taramadaki
hasar listesiyle. `validate="one_to_one"` iki tarafta da tekrar varsa hata verir.

In [47]:
matched = listings.merge(damage, on=["ad_id", "snap"], how="inner", validate="one_to_one")
print("Eşleşen ilan:", tr_int(len(matched)), "/", tr_int(len(listings)))

Eşleşen ilan: 29.988 / 29.988


Şimdi sınama: her parça için üç bayrağı ham cevaptan yeniden kuruyoruz ("değişmiş" cevabı değişen bayrağı, "boyalı"
boyalı bayrağı, "lokal boyalı" lokal bayrağı) ve veritabanının bayraklarıyla, boş bayrağı 0 sayarak (`load_clean`
gibi), karşılaştırıyoruz. 13 parça × 3 bayrak = 39 karşılaştırma; tek bir ilanda bile fark çıkarsa hücre durur.
Veritabanında bayrak kolonlarının öneki çoğunlukla durum kolonunun adından gelir (`door_fl_status` → `door_fl_...`);
tavan, kaput ve bagajın öneki Türkçe.

In [48]:
FLAG_PREFIX = {"roof_status": "tavan", "engine_hood_status": "kaput", "trunk_lid_status": "bagaj"}
ANSWER_OF_FLAG = {"degisen": "changed", "boyali": "painted", "lokal": "local_painted"}

n_flags_checked = 0
for status_column in STATUS_COLUMNS:
    prefix = FLAG_PREFIX.get(status_column, status_column.removesuffix("_status"))
    for flag_suffix, answer in ANSWER_OF_FLAG.items():
        db_flag = matched[f"{prefix}_{flag_suffix}"].fillna(0).astype(int)
        rebuilt_flag = (matched[status_column] == answer).astype(int)
        assert (db_flag == rebuilt_flag).all(), f"{status_column} / {flag_suffix}: ham cevap veritabanından farklı"
        n_flags_checked += 1
print(n_flags_checked, "bayrağın hepsi, bütün ilanlarda veritabanıyla aynı.")

39 bayrağın hepsi, bütün ilanlarda veritabanıyla aynı.


Sorunun başında söylenen de sınanabilir: veritabanında bir parçanın üç bayrağı birden boşsa, ham cevap
"belirtilmemiş" mi? Öyleyse ayrım veritabanında duruyor, 0'a çevrilmesi `load_clean`'de oluyor.

In [49]:
for status_column in STATUS_COLUMNS:
    prefix = FLAG_PREFIX.get(status_column, status_column.removesuffix("_status"))
    flag_columns = [f"{prefix}_{flag_suffix}" for flag_suffix in ANSWER_OF_FLAG]
    all_flags_empty = matched[flag_columns].isna().all(axis=1)
    is_unspecified = matched[status_column] == "unspecified"
    assert (all_flags_empty == is_unspecified).all(), f"{status_column}: boş bayrak ile belirtilmemiş farklı"
print("13 parçanın hepsinde: üç bayrak birden boş <=> ham cevap 'belirtilmemiş'.")

13 parçanın hepsinde: üç bayrak birden boş <=> ham cevap 'belirtilmemiş'.


#### 1.3.5 Sayım

Artık sayabiliriz. Önce bütün parça cevaplarının dağılımı, sonra ilan başına kaç parçanın belirtilmemiş olduğu.

In [50]:
answers = matched[STATUS_COLUMNS]
n_panels = answers.notna().sum().sum()
n_unspecified = (answers == "unspecified").sum().sum()
print(f"Parça cevabı: {tr_int(n_panels)} · belirtilmemiş: {tr_int(n_unspecified)} "
      f"(%{100 * n_unspecified / n_panels:.1f})")

site_label = {code: label for label, code in status_map.items()}   # kod -> sitedeki cevap, yalnız gösterim için
answers.stack().value_counts().rename(index=site_label).rename_axis("cevap").rename("parça").to_frame()

Parça cevabı: 389.844 · belirtilmemiş: 59.651 (%15.3)


,parça
cevap,
Orjinal,256281
Belirtilmemiş,59651
Boyalı,42719
Lokal boyalı,17806
Değişmiş,13387


In [51]:
unspecified_per_listing = (answers == "unspecified").sum(axis=1)

unspecified_distribution = unspecified_per_listing.value_counts().sort_index()
unspecified_distribution.rename_axis("belirtilmemiş parça sayısı").rename("ilan").to_frame().T

belirtilmemiş parça sayısı,0,1,2,3,4,5,6,7,8,9,10,11,12,13
ilan,23752,131,352,107,107,146,167,269,443,659,787,944,889,1235


**Ne görüyoruz.** 389.844 parça cevabının 59.651'i (%15.3) belirtilmemiş; orijinalden sonra en sık cevap bu.
İlanlara göre dağılım iki uçlu: 23.752 ilanda hiç belirtilmemiş parça yok, 1.235 ilanda 13 parçanın hepsi
belirtilmemiş (satıcı hasar şemasını hiç doldurmamış), en az bir belirtilmemiş parçası olan ilan 6.236. Arada
kalanların çoğunda 8-12 parça belirtilmemiş: satıcı yalnız birkaç parçayı işaretlemiş. Tahmin modeli bu ilanların
belirtilmemiş parçalarını orijinal görüyor; hasarlı olup bunu yazmamış bir satıcının aracı tahmin modeline hasarsız
görünür.

#### 1.3.6 Kayıtla karşılaştırma

In [52]:
record = read_record("01_unspecified_panels")["error_drivers"]["unspecified"]
structure = record["structure"]

assert record["matched_listings"] == len(matched) and record["listings"] == len(listings)
assert record["duplicates_skipped"] == duplicated_pages.sum()
assert record["n_panels"] == n_panels and record["unspecified_panels"] == n_unspecified
assert record["unspecified_pct"] == round(100 * n_unspecified / n_panels, 1)
assert record["all_unspecified"] == (unspecified_per_listing == len(STATUS_COLUMNS)).sum()
assert record["at_least_one"] == (unspecified_per_listing > 0).sum()
assert structure["panel"] == len(STATUS_COLUMNS) and structure["flags"] == n_flags_checked
assert structure["n_answers"] == len(set(status_map.values()))
print(f"Kayıtla aynı: {tr_int(len(matched))} ilan eşleşti; {tr_int(n_unspecified)} belirtilmemiş parça "
      f"(%{record['unspecified_pct']}); {tr_int(record['at_least_one'])} ilanda en az bir, "
      f"{tr_int(record['all_unspecified'])} ilanda hepsi.")

Kayıtla aynı: 29.988 ilan eşleşti; 59.651 belirtilmemiş parça (%15.3); 6.236 ilanda en az bir, 1.235 ilanda hepsi.


### Bölüm 1'in özeti

- **45.159 tarama satırı → 29.988 ilan.** Her ilanın en son satırı kullanılıyor; fiyat geçmişi (4.703 ilanın fiyatı
  değişmiş) tahmin modeline girmiyor.
- **Kapsam:** yalnız TR plakalı ilanlar (76 ilan dışarıda); fiyat 6.500.000 ₺'de sağdan kesik, yaş en fazla 21,
  yalnız otomobil kategorisi, elektrikli yok.
- **Sızıntı riski:** yeni numarayla yeniden yayımlanmış aynı araç ilanların %0.46'sı (katı tanım) ile %4.26'sı (fiyat
  hariç) arasında.
- **Motor:** hacim = aralığın üst sınırı, güç = iki sınırın ortalaması; iki kural da veriyle en uyumlu aday.
- **Hasar:** parça cevaplarının %15.3'ü "belirtilmemiş" ve orijinal sayılıyor; 1.235 ilanda 13 parçanın hepsi
  belirtilmemiş.

Bundan sonraki bölümler `listings` tablosunun bu hâliyle çalışıyor.